### Helper Functions for Open Hours Calculation

These functions will parse the `open_hours` data from your JSON files and calculate the total open hours for each entry, following your specified rules:

*   Parses time strings (e.g., "7:30 AM–10 PM") into a duration in hours.
*   Calculates the total open hours across all available days.
*   If data is present for only one day, it multiplies that day's open hours by 6.
*   Missing `total_open_hours` (due to no `open_hours` data) will be filled with the mean later.

In [37]:
from datetime import datetime, timedelta
import pandas as pd
import os
import json
import re

def parse_time_string(time_str):
    """
    Parses a time string like '7:45 AM' or '10 PM' into a timedelta object
    representing time from midnight.
    """
    # Normalize input to handle different AM/PM cases and remove extra spaces
    time_str = time_str.replace(' AM', ' AM').replace(' PM', ' PM').strip().replace('.', '')

    # Try different formats, being flexible with optional minutes
    formats = ["%I:%M %p", "%I %p"]
    for fmt in formats:
        try:
            dt_object = datetime.strptime(time_str, fmt)
            return timedelta(hours=dt_object.hour, minutes=dt_object.minute)
        except ValueError:
            pass
    raise ValueError(f"Could not parse time string: {time_str}")

def calculate_duration_from_intervals(intervals):
    """
    Calculates the total duration in hours from a list of time intervals
    like ['7:45 AM–10 PM']. Handles intervals spanning midnight.
    """
    total_duration_minutes = 0
    if not intervals or not isinstance(intervals, list):
        return 0.0

    for interval_str in intervals:
        if not isinstance(interval_str, str):
            continue

        # Standardize dash characters
        interval_str = interval_str.replace('–', '-').replace('—', '-')

        if 'Open 24 hours' in interval_str or '24 hours' in interval_str.lower():
            total_duration_minutes += 24 * 60
            continue
        if 'Closed' in interval_str:
            continue

        parts = interval_str.split('-')
        if len(parts) != 2:
            continue

        try:
            start_td = parse_time_string(parts[0].strip())
            end_td = parse_time_string(parts[1].strip())

            duration = end_td - start_td
            if duration < timedelta(0):
                duration += timedelta(days=1)

            total_duration_minutes += duration.total_seconds() / 60
        except ValueError:
            continue

    return total_duration_minutes / 60.0

def calculate_total_open_hours(open_hours_data):
    """
    Calculates the total open hours based on the provided open_hours dictionary.
    Applies special rules for single-day data.
    """
    if not open_hours_data or not isinstance(open_hours_data, dict):
        return float('nan')

    daily_hours = {}
    for day, intervals in open_hours_data.items():
        daily_hours[day] = calculate_duration_from_intervals(intervals)

    total_sum_hours = sum(daily_hours.values())
    num_non_zero_days = sum(1 for h in daily_hours.values() if h > 0)

    if num_non_zero_days == 1:
        return total_sum_hours * 6

    return total_sum_hours

### Mount Google Drive

To access files stored in your Google Drive, you need to mount it. This cell will prompt you to authorize Google Colab to access your Drive.

In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Path Troubleshooting
Run this cell to see exactly what folders are in your 'My Drive'. This will help us correct the `json_directory_path`.

In [39]:
import os

drive_root = '/content/drive/My Drive/'
if os.path.exists(drive_root):
    print(f"Folders found in {drive_root}:")
    folders = [f for f in os.listdir(drive_root) if os.path.isdir(os.path.join(drive_root, f))]
    for folder in sorted(folders):
        print(f" - {folder}")
else:
    print("Google Drive is not mounted. Please run the 'drive.mount' cell first.")

Folders found in /content/drive/My Drive/:
 - ktm_all_all


### Specify JSON Directory and Process Files

Enter the path to your directory containing the JSON files below. The code will then:
1.  Find all `.json` files in the specified directory.
2.  Load each JSON file and process its data.
3.  Apply the `calculate_total_open_hours` function to each entry.
4.  Combine all processed data into a single Pandas DataFrame.
5.  Fill any `NaN` values in `total_open_hours` with the mean of that column, as per your request.
6.  Save the consolidated data into a CSV file.

In [42]:
import os
import json
import pandas as pd
import ast
import numpy as np

datasetName = "all"
json_directory_path = f"/content/drive/My Drive/ktm_{datasetName}_all/"
output_dir = os.path.join(json_directory_path, 'nb_output')
os.makedirs(output_dir, exist_ok=True)
output_csv_path = os.path.join(output_dir, f'combined_{datasetName}s.csv')

print(f"Searching for raw JSON files in: {json_directory_path}")
all_data = []
if os.path.isdir(json_directory_path):
    json_files = [f for f in os.listdir(json_directory_path) if f.endswith('.json')]
    for filename in json_files:
        filepath = os.path.join(json_directory_path, filename)
        try:
            with open(filepath, 'r') as f:
                file_data = json.load(f)
                if isinstance(file_data, dict): file_data = [file_data]
                for item in file_data:
                    if 'cid' in item: del item['cid']
                    # Preserve nested structures for later DB normalization
                    all_data.append(item)
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    if all_data:
        df = pd.DataFrame(all_data).drop_duplicates(subset=['place_id']).reset_index(drop=True)
        print(f"Loaded {len(df)} entries with raw nested columns available.")
    else:
        print("No data found.")
else:
    print("Directory not found.")

Searching for raw JSON files in: /content/drive/My Drive/ktm_all_all/
No data found.


### Plot 1: Overall Average Popularity by Hour

In [41]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

# Assuming datasetName and output_csv_path are defined in a previous cell (like cb00271a)
# The output_csv_path should be available from the previous cell (cb00271a) after it is run
# We will use output_csv_path directly to load the data.

print(f"Loading data from: {output_csv_path}")
df_popular_times = pd.read_csv(output_csv_path)

# Function to safely parse the 'popular_times' string
def parse_popular_times(pt_str):
    if pd.isna(pt_str) or pt_str == "{}":
        return {}
    try:
        # Some strings are saved as double-encoded or JSON strings, ast handles literal dict strings better
        if isinstance(pt_str, str) and pt_str.startswith('{'):
            return ast.literal_eval(pt_str)
        return {}
    except (ValueError, SyntaxError):
        try:
            return json.loads(pt_str.replace("'", "\""))
        except:
            return {}

# Apply the parsing function and flatten the data
flattened_popular_times = []
for index, row in df_popular_times.iterrows():
    place_id = row.get('place_id', 'unknown')
    pt_data = row.get('popular_times', '{}')
    parsed_pt = parse_popular_times(pt_data)

    if isinstance(parsed_pt, dict):
        for day, hourly_data in parsed_pt.items():
            if isinstance(hourly_data, dict):
                for hour_str, popularity in hourly_data.items():
                    flattened_popular_times.append({
                        'place_id': place_id,
                        'day': day,
                        'hour': int(hour_str),
                        'popularity': popularity
                    })

popular_times_df = pd.DataFrame(flattened_popular_times)

if not popular_times_df.empty:
    # Define the order of days for consistent plotting
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    popular_times_df['day'] = pd.Categorical(popular_times_df['day'], categories=day_order, ordered=True)
    display(popular_times_df.head())
    print(f"Total entries in flattened popular times data: {len(popular_times_df)}")
else:
    print("Warning: No popular times data was found or parsed.")

Loading data from: /content/drive/My Drive/ktm_all_all/nb_output/combined_alls.csv


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/ktm_all_all/nb_output/combined_alls.csv'

### Plot 2: Hourly Popularity Trends by Day of the Week

In [ ]:
# Calculate overall average popularity for each hour
overall_hourly_avg = popular_times_df.groupby('hour')['popularity'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(x='hour', y='popularity', data=overall_hourly_avg, marker='o')
plt.title('Overall Average Popularity by Hour Across All Cafes/Restaurants')
plt.xlabel('Hour of Day (24-hour format)')
plt.ylabel('Average Popularity (%)')
plt.xticks(range(0, 24)) # Ensure all hours are displayed on x-axis
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate average popularity per hour per day
daily_hourly_avg = popular_times_df.groupby(['day', 'hour'])['popularity'].mean().reset_index()

plt.figure(figsize=(15, 8))
sns.lineplot(x='hour', y='popularity', hue='day', data=daily_hourly_avg, marker='o')
plt.title('Average Hourly Popularity by Day of the Week')
plt.xlabel('Hour of Day (24-hour format)')
plt.ylabel('Average Popularity (%)')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Day of Week', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Visualizing Popular Times

To visualize the 'popular_times' data, we first need to extract and flatten the nested JSON strings into a tabular format. Then, we can calculate and plot the average popularity across different hours and days.

### Data Type Analysis

Now that the data is consolidated, let's analyze the data types of each column to understand their nature (categorical, numerical, discrete). This will involve:

1.  **Displaying DataFrame info**: To get an overview of column names and their inferred types.
2.  **Descriptive Statistics**: For numerical columns.
3.  **Unique Values/Value Counts**: For categorical or discrete columns to understand their distribution.
4.  **Identifying Data Types**: Based on the above, categorizing columns as numerical (continuous/discrete) or categorical.

In [ ]:
import pandas as pd
import numpy as np

# Data Type Analysis
print("\n--- DataFrame Information ---")
df.info()

print("\n--- Descriptive Statistics ---")
display(df.describe())

print("\n--- Analysis of Column Data Types ---")

for col in df.columns:
    # Check if the column contains lists or dicts which are unhashable
    # We convert to string temporarily for unique counting to avoid TypeErrors
    is_complex = df[col].apply(lambda x: isinstance(x, (list, dict))).any()

    if is_complex:
        num_unique = df[col].astype(str).nunique()
        print(f"- Column '{col}': **Complex (List/Dict)**. Unique string signatures: {num_unique}")
        continue

    dtype = df[col].dtype
    num_unique = df[col].nunique()

    if pd.api.types.is_numeric_dtype(dtype):
        if num_unique < 20:
            print(f"- Column '{col}' ({dtype}): Likely **Discrete Numerical**. Unique values: {num_unique}")
        else:
            print(f"- Column '{col}' ({dtype}): Likely **Continuous Numerical**.")
    elif pd.api.types.is_object_dtype(dtype) or pd.api.types.is_bool_dtype(dtype):
        if num_unique < 50:
            print(f"- Column '{col}' ({dtype}): Likely **Categorical**. Unique values: {num_unique}")
        else:
            print(f"- Column '{col}' ({dtype}): Likely **Text/Identifier**. Unique values: {num_unique}")

In [ ]:
def flatten_about(df):
    if 'about' not in df.columns:
        print("'about' column not found. Skipping flattening.")
        return df

    about_data = []
    for _, row in df.iterrows():
        flattened = {}
        item_about = row.get('about')

        if isinstance(item_about, list):
            for section in item_about:
                section_name = section.get('name', 'unknown')
                options = section.get('options', [])
                for opt in options:
                    opt_name = opt.get('name', 'unknown')
                    col_name = f"about_{section_name}_{opt_name}"
                    flattened[col_name] = 1 if opt.get('enabled') else 0
        about_data.append(flattened)

    about_df = pd.DataFrame(about_data).fillna(0).astype(int)
    # Ensure we return the combined dataframe correctly
    return pd.concat([df.drop(columns=['about']).reset_index(drop=True), about_df], axis=1)

# 1. Create reviews_count from user_reviews length AND update the main df
if 'user_reviews' in df.columns:
    df['reviews_count_from_list'] = df['user_reviews'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# 2. Flatten the 'about' column and re-assign back to df
df = flatten_about(df)

print(f"New shape: {df.shape}")
print(f"Columns available: {'reviews_count_from_list' in df.columns}")
display(df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Uniqueness and Missing Values ---
unique_stats = []
for col in df.columns:
    dtype = df[col].dtype
    missing = df[col].isna().sum()
    unique = df[col].astype(str).nunique()
    unique_stats.append({'column': col, 'dtype': dtype, 'unique_count': unique, 'missing_values': missing})

unique_df = pd.DataFrame(unique_stats)
print("--- Column Uniqueness Summary ---")
display(unique_df.sort_values(by='unique_count', ascending=False))

# --- 2. Visualization of Key Numerical Distributions ---
# Check which columns actually exist before plotting to avoid KeyError
requested_num_cols = ['review_count', 'review_rating', 'total_open_hours', 'reviews_count_from_list']
num_cols = [c for c in requested_num_cols if c in df.columns]

if num_cols:
    n_plots = len(num_cols)
    rows = (n_plots + 1) // 2
    plt.figure(figsize=(15, 5 * rows))
    for i, col in enumerate(num_cols, 1):
        plt.subplot(rows, 2, i)
        sns.histplot(df[col].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns available for distribution plots.")

# --- 3. Top Categories Visualization ---
if 'category' in df.columns:
    plt.figure(figsize=(10, 6))
    df['category'].value_counts().head(15).plot(kind='bar')
    plt.title('Top 15 Categories')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45, ha='right')
    plt.show()

print("\n--- GNN Potential Feature Significance Analysis Complete ---")

In [ ]:
import pandas as pd
import numpy as np

def generate_full_summary(df):
    summary_list = []
    for col in df.columns:
        # Basic Stats
        unique_val = df[col].astype(str).nunique()
        missing_val = df[col].isna().sum()

        # Initialize stats
        mean_val, median_val, mode_val = "N/A", "N/A", "N/A"

        # Detect if column contains complex types (lists/dicts)
        is_complex = df[col].apply(lambda x: isinstance(x, (list, dict))).any()

        if pd.api.types.is_numeric_dtype(df[col]):
            mean_val = round(df[col].mean(), 2)
            median_val = round(df[col].median(), 2)
            mode_series = df[col].mode()
            mode_val = mode_series.iloc[0] if not mode_series.empty else "N/A"
        elif not is_complex:
            # Only calculate mode for simple categorical/text columns
            mode_series = df[col].mode()
            mode_val = mode_series.iloc[0] if not mode_series.empty else "N/A"
        else:
            mode_val = "[Complex Data Type]"

        summary_list.append({
            "Column": col,
            "Uniqueness": unique_val,
            "Missing": missing_val,
            "Mean": mean_val,
            "Median": median_val,
            "Mode/Top": mode_val
        })

    return pd.DataFrame(summary_list)

final_summary_df = generate_full_summary(df)
print("--- Comprehensive Column-wise Summary ---")
display(final_summary_df)

# Quick print for significant features
print("\nSnapshot of GNN-Ready Numerical Features:")
display(df[['review_count', 'review_rating', 'total_open_hours', 'reviews_count_from_list']].describe())

### Comprehensive Dataset Summary
Below is the statistical summary of all columns, including uniqueness, missing values, and central tendencies. You can also download this summary as a CSV.

In [ ]:
from google.colab import data_table
import os

# 1. Save summary to CSV
summary_csv_path = os.path.join(output_dir, 'dataset_summary.csv')
final_summary_df.to_csv(summary_csv_path, index=False)
print(f"Summary CSV saved to: {summary_csv_path}")

# 2. Display Interactive Table
data_table.enable_dataframe_formatter()
display(final_summary_df)

# 3. Generate Markdown for display
print("\n--- Markdown Table Representation ---")
print(final_summary_df.to_markdown(index=False))

In [ ]:
# ### Summary Generated
# The summary CSV has been created and will be backed up to Drive in the final step.
print("Summary prepared for final backup.")

### Export Summary to Markdown
This cell exports the `final_summary_df` to a markdown file and triggers a download.

In [ ]:
import os

# Define markdown file path
md_file_path = os.path.join(output_dir, 'dataset_summary.md')

# Convert dataframe to markdown string
md_content = "# Dataset Summary\n\n" + final_summary_df.to_markdown(index=False)

# Write to file
with open(md_file_path, 'w') as f:
    f.write(md_content)

print(f"Markdown summary prepared at: {md_file_path}")

### Data Cleaning & SQLite Database Creation
In this step, we drop low-variance binary features and normalize the complex nested data into a relational SQLite structure.

In [ ]:
import sqlite3
import os
import pandas as pd

# 1. Setup Database Path
db_path = os.path.join(output_dir, f'ktm_{datasetName}.db')
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Helper to clear old tables to ensure fresh schema with proper keys
cursor.execute("DROP TABLE IF EXISTS cafes_main")
cursor.execute("DROP TABLE IF EXISTS reviews")
cursor.execute("DROP TABLE IF EXISTS images")

# --- 1. Main Table (cafes_main) ---
complex_cols = ['user_reviews', 'user_reviews_extended', 'images', 'owner', 'open_hours', 'popular_times', 'menu']
main_db_df = df.drop(columns=[c for c in complex_cols if c in df.columns]).copy()

# FIX: Ensure no overflow for large integers and handle complex types
for col in main_db_df.columns:
    # Cast large integers or problematic types to string to avoid SQLite limits (~9e18)
    if main_db_df[col].dtype == 'object' or pd.api.types.is_integer_dtype(main_db_df[col]):
        if main_db_df[col].apply(lambda x: isinstance(x, (int, float)) and abs(x) > 9e18).any():
            main_db_df[col] = main_db_df[col].astype(str)

    # Ensure remaining complex types (lists/dicts) are strings
    if main_db_df[col].apply(lambda x: isinstance(x, (list, dict))).any():
        main_db_df[col] = main_db_df[col].astype(str)

main_db_df.to_sql('cafes_main', conn, if_exists='replace', index=False)
cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_place_id ON cafes_main (place_id)")
print("Table 'cafes_main' created safely.")

# --- 2. Combined Reviews Table ---
all_reviews = []
for _, row in df.iterrows():
    revs = row.get('user_reviews', []) if isinstance(row.get('user_reviews'), list) else []
    ext = row.get('user_reviews_extended', []) if isinstance(row.get('user_reviews_extended'), list) else []
    for r in revs + ext:
        if isinstance(r, dict):
            r_copy = r.copy()
            r_copy['place_id'] = row['place_id']
            all_reviews.append(r_copy)

if all_reviews:
    rev_df = pd.DataFrame(all_reviews).astype(str)
    rev_df.to_sql('reviews', conn, if_exists='replace', index=False)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_rev_place_id ON reviews (place_id)")
    print(f"Table 'reviews' created with {len(all_reviews)} entries.")

# --- 3. Images Table ---
images_list = []
for _, row in df.iterrows():
    imgs = row.get('images', [])
    if isinstance(imgs, list):
        for img in imgs:
            if isinstance(img, dict):
                img_copy = img.copy()
                img_copy['place_id'] = row['place_id']
                images_list.append(img_copy)

if images_list:
    img_df = pd.DataFrame(images_list).astype(str)
    img_df.to_sql('images', conn, if_exists='replace', index=False)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_img_place_id ON images (place_id)")
    print(f"Table 'images' created with {len(images_list)} entries.")

conn.commit()
conn.close()
print(f"Relational SQLite Database finalized at: {db_path}")

In [ ]:
import pandas as pd
import numpy as np

# 1. Identify all already-flattened 'about_' columns
about_cols = [col for col in df.columns if col.startswith('about_')]

# 2. Calculate counts and percentages for each column
about_stats = []
if not about_cols:
    print("Warning: No 'about_' columns found. Please ensure cell 99e7e9de was executed correctly.")
    stats_df = pd.DataFrame(columns=['column', 'count_0', 'count_1', 'percent_1'])
else:
    for col in about_cols:
        # Ensure we are working with numeric data for these columns
        counts = df[col].value_counts()
        num_zeros = counts.get(0, 0)
        num_ones = counts.get(1, 0)
        pct_ones = (num_ones / len(df)) * 100

        about_stats.append({
            'column': col,
            'count_0': num_zeros,
            'count_1': num_ones,
            'percent_1': round(pct_ones, 2)
        })
    stats_df = pd.DataFrame(about_stats)

# 3. Suggest significance if data exists
if not stats_df.empty:
    significance_threshold = 20.0
    significant_cols = stats_df[stats_df['percent_1'] >= significance_threshold]['column'].tolist()
    insignificant_cols = stats_df[stats_df['percent_1'] < significance_threshold]['column'].tolist()

    print(f"Total 'about_' columns analyzed: {len(about_cols)}")
    print(f"Columns with >= {significance_threshold}% ones: {len(significant_cols)}")
    print(f"Columns with < {significance_threshold}% ones (potentially insignificant): {len(insignificant_cols)}")

    # 4. Show the detailed breakdown
    print("\n--- Top 20 'about_' Columns by Frequency of 1s ---")
    display(stats_df.sort_values(by='percent_1', ascending=False).head(20))

    # Drop columns that are below the significance threshold
    if insignificant_cols:
        print(f"\nDropping {len(insignificant_cols)} columns with less than {significance_threshold}% ones.")
        df = df.drop(columns=insignificant_cols)

    # Optional: Drop columns that are completely constant (0 variance) - this is already handled by insignificant_cols if threshold is low enough
    # const_about_cols = [col for col in about_cols if df[col].nunique() <= 1]
    # if const_about_cols:
    #     print(f"\nDropping {len(const_about_cols)} columns with zero variance.")
    #     df = df.drop(columns=const_about_cols)
else:
    print("Skipping significance analysis as no 'about_' columns were detected.")

print(f"\nNew dataframe shape: {df.shape}")

### Updating the Database
Since we've refined the features, let's update the SQLite database with the cleaner main table.

In [ ]:
import sqlite3
import os
import numpy as np

# Re-define path using the directory path established earlier
db_path = os.path.join(output_dir, f'ktm_{datasetName}.db')

# Re-connect to update the main table
conn = sqlite3.connect(db_path)

# Identify columns to exclude from the main table (relational data held in other tables)
complex_cols = ['user_reviews', 'user_reviews_extended', 'images', 'owner', 'open_hours', 'popular_times', 'menu']
clean_main_df = df.drop(columns=[c for c in complex_cols if c in df.columns]).copy()

# Fix OverflowError: Convert large integers or problematic types to string
for col in clean_main_df.columns:
    # If the column is object type or has very large integers, cast to string
    if clean_main_df[col].dtype == 'object' or pd.api.types.is_integer_dtype(clean_main_df[col]):
        # Check if any value is too large for SQLite (max signed 64-bit int is ~9e18)
        if clean_main_df[col].apply(lambda x: isinstance(x, int) and abs(x) > 9000000000000000000).any():
            clean_main_df[col] = clean_main_df[col].astype(str)

    # Ensure remaining complex types are strings
    if clean_main_df[col].apply(lambda x: isinstance(x, (list, dict))).any():
        clean_main_df[col] = clean_main_df[col].astype(str)

# Update the 'cafes_main' table
clean_main_df.to_sql('cafes_main', conn, if_exists='replace', index=False)
conn.close()

print("SQLite database 'cafes_main' table has been updated successfully.")

In [ ]:
import pandas as pd
import json
import sqlite3
import os

def flatten_address(df):
    """
    Reimplemented from the ground up to parse 'complete_address'
    and normalize address fields into separate columns.
    """
    # 1. Initialize a list to hold the new address dictionaries
    parsed_addresses = []

    # 2. Iterate through the dataframe to extract address components
    for _, row in df.iterrows():
        raw_addr = row.get('complete_address')
        addr_dict = {}

        if pd.notna(raw_addr):
            try:
                # Attempt to parse if it's a string representation of a dict
                if isinstance(raw_addr, str):
                    # Handle potential single quote vs double quote issues for JSON
                    addr_dict = json.loads(raw_addr.replace("'", "\""))
                elif isinstance(raw_addr, dict):
                    addr_dict = raw_addr
            except (json.JSONDecodeError, ValueError):
                addr_dict = {}

        # 3. Define standardized fields and extract values
        parsed_addresses.append({
            'address_borough': addr_dict.get('borough', ''),
            'address_street': addr_dict.get('street', ''),
            'address_city': addr_dict.get('city', ''),
            'address_postal_code': addr_dict.get('postal_code', ''),
            'address_state': addr_dict.get('state', ''),
            'address_country': addr_dict.get('country', '')
        })

    # 4. Create new columns from the list of dicts
    addr_df = pd.DataFrame(parsed_addresses)

    # 5. Combine with original dataframe (dropping the original nested column)
    df_cleaned = df.drop(columns=['complete_address']) if 'complete_address' in df.columns else df

    # Reset indexes to ensure perfect alignment during concat
    df_cleaned = df_cleaned.reset_index(drop=True)
    addr_df = addr_df.reset_index(drop=True)

    return pd.concat([df_cleaned, addr_df], axis=1)

# --- Fix: Drop existing address columns before flattening to prevent duplication ---
address_cols_to_drop = ['address_borough', 'address_street', 'address_city', 'address_postal_code', 'address_state', 'address_country']
df = df.drop(columns=[col for col in address_cols_to_drop if col in df.columns], errors='ignore')

# --- Execution ---
# Apply the fresh implementation
df = flatten_address(df)

# Update CSV (now saves to output_dir directly)
output_csv_path = os.path.join(output_dir, f'combined_{datasetName}s.csv')
df.to_csv(output_csv_path, index=False)

# Update SQLite Database (now uses output_dir directly)
db_path = os.path.join(output_dir, f'ktm_{datasetName}.db')
conn = sqlite3.connect(db_path)

# Filter for main table columns only
relational_cols = ['user_reviews', 'user_reviews_extended', 'images', 'owner', 'open_hours', 'popular_times', 'menu']
clean_db_df = df.drop(columns=[c for c in relational_cols if c in df.columns]).copy()

# Stringify any remaining complex objects for SQLite compatibility
for col in clean_db_df.columns:
    if clean_db_df[col].apply(lambda x: isinstance(x, (list, dict))).any():
        clean_db_df[col] = clean_db_df[col].astype(str)

clean_db_df.to_sql('cafes_main', conn, if_exists='replace', index=False)
conn.close()

print("Reimplementation successful: Address flattened and database updated.")
display(df[['title', 'address_city', 'address_street']].head())

In [ ]:
import sqlite3
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Verification of row counts in the new normalized tables
tables = ['open_hours_detailed', 'popular_times_detailed', 'categories_detailed', 'reviews_detailed']
print("--- Database Table Row Counts ---")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table}: {count} rows")

conn.close()
print("\nDatabase verification complete.")

### Natural Language Summary of the Dataset

Based on the analysis above, here is a summary of your consolidated and flattened dataset:

1. **Data Volume & Deduplication**: After removing duplicates by `place_id`, we have **7,856 unique nodes** (cafes/restaurants). Originally, the data had over 26,000 entries, indicating significant overlap between your JSON files.

2. **Operational Patterns (`total_open_hours`)**: The mean open time is approximately **92.23 hours per week**. This is a strong continuous feature for your GNN, helping differentiate between standard eateries and 24/7 or late-night spots.

3. **Social Proof & Engagement**:
   - The **Mean Review Rating** is **3.90**, suggesting a generally positive sentiment, but with a standard deviation that shows enough variance to be a useful label or feature.
   - We created `reviews_count_from_list` which directly measures the raw data depth available for each node (the actual number of review objects found).

4. **Feature Richness (The 'About' Flattening)**: The dataset expanded to **190 columns**. This is because the `about` field contained diverse attributes like 'Takeaway', 'Outdoor Seating', and 'Good for kids'. These binary flags (0 or 1) are perfect for a GNN to understand node similarity (e.g., nodes sharing many 'enabled' flags are likely similar).

5. **Category Diversity**: There are **321 unique sub-categories**, though the top 15 (like 'Restaurant', 'Cafe', 'Coffee shop') dominate. This high cardinality suggests you might want to use embeddings or one-hot encoding for the top categories when feeding them into your GNN.

6. **Missing Data**: Most core fields are complete. Fields like `reservations` and `order_online` have many nulls, which effectively tells the GNN that these services are likely 'not offered' at those specific locations.

### Detailed Time Parsing for SQLite
This step extracts specific start and end hours/minutes from the `open_hours` and `popular_times` fields, converting them to a 24-hour format for better relational querying.

In [ ]:
import sqlite3
import re
import pandas as pd
import json
import ast

def time_to_24h(time_str):
    if not isinstance(time_str, str): return None, None
    time_str = time_str.replace('\u202f', ' ').strip().upper()
    match = re.search(r'(\d+)(?::(\d+))?\s*(AM|PM)?', time_str)
    if not match: return None, None
    hour = int(match.group(1))
    minute = int(match.group(2)) if match.group(2) else 0
    period = match.group(3)
    if period == 'PM' and hour < 12: hour += 12
    if period == 'AM' and hour == 12: hour = 0
    return hour, minute

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# --- 1. Process Open Hours (with improved parsing) ---
open_hours_rows = []
for _, row in df.iterrows():
    oh = row.get('open_hours')
    if pd.isna(oh): continue
    if isinstance(oh, str):
        try: oh = ast.literal_eval(oh)
        except:
            try: oh = json.loads(oh.replace("'", "\""))
            except: continue

    if isinstance(oh, dict):
        for day, intervals in oh.items():
            if not isinstance(intervals, list): continue
            for interval in intervals:
                if not isinstance(interval, str): continue
                if '-' in interval or '–' in interval:
                    parts = re.split(r'[-–]', interval)
                    s_h, s_m = time_to_24h(parts[0])
                    e_h, e_m = time_to_24h(parts[1])
                    if s_h is not None:
                        open_hours_rows.append((row['place_id'], day, s_h, s_m, e_h, e_m))

cursor.execute("DROP TABLE IF EXISTS open_hours_detailed")
cursor.execute("CREATE TABLE open_hours_detailed (place_id TEXT, day TEXT, start_hour INTEGER, start_minute INTEGER, end_hour INTEGER, end_minute INTEGER)")
if open_hours_rows:
    cursor.executemany("INSERT INTO open_hours_detailed VALUES (?,?,?,?,?,?)", open_hours_rows)
    print(f"Inserted {len(open_hours_rows)} rows into open_hours_detailed.")

# --- 2. Process Popular Times ---
popular_rows = []
for _, row in df.iterrows():
    pt = row.get('popular_times')
    if pd.isna(pt): continue
    if isinstance(pt, str):
        try: pt = ast.literal_eval(pt)
        except: continue
    if isinstance(pt, dict):
        for day, hours in pt.items():
            if isinstance(hours, dict):
                for hr, val in hours.items():
                    popular_rows.append((row['place_id'], day, int(hr), val))

cursor.execute("DROP TABLE IF EXISTS popular_times_detailed")
cursor.execute("CREATE TABLE popular_times_detailed (place_id TEXT, day TEXT, hour INTEGER, popularity_percent INTEGER)")
if popular_rows:
    cursor.executemany("INSERT INTO popular_times_detailed VALUES (?,?,?,?)", popular_rows)
    print(f"Inserted {len(popular_rows)} rows into popular_times_detailed.")

# --- 3. Process Categories ---
cat_rows = []
for _, row in df.iterrows():
    cats = row.get('categories')
    if pd.isna(cats): continue
    if isinstance(cats, str):
        try: cats = ast.literal_eval(cats)
        except: cats = [cats]
    if isinstance(cats, list):
        for c in cats:
            cat_rows.append((row['place_id'], str(c)))

cursor.execute("DROP TABLE IF EXISTS categories_detailed")
cursor.execute("CREATE TABLE categories_detailed (place_id TEXT, category_name TEXT)")
if cat_rows:
    cursor.executemany("INSERT INTO categories_detailed VALUES (?,?)", cat_rows)
    print(f"Inserted {len(cat_rows)} rows into categories_detailed.")

# --- 4. Process Reviews ---
review_rows = []
for _, row in df.iterrows():
    revs = row.get('user_reviews')
    if pd.isna(revs): continue
    if isinstance(revs, str):
        try: revs = ast.literal_eval(revs)
        except: revs = []
    if isinstance(revs, list):
        for r in revs:
            if isinstance(r, dict):
                review_rows.append((row['place_id'], r.get('Name'), r.get('Rating'), r.get('Description'), r.get('When')))

cursor.execute("DROP TABLE IF EXISTS reviews_detailed")
cursor.execute("CREATE TABLE reviews_detailed (place_id TEXT, user_name TEXT, rating REAL, text TEXT, time TEXT)")
if review_rows:
    cursor.executemany("INSERT INTO reviews_detailed VALUES (?,?,?,?,?)", review_rows)
    print(f"Inserted {len(review_rows)} rows into reviews_detailed.")

conn.commit()
conn.close()
print("Database normalization complete.")

In [ ]:
import sqlite3
import pandas as pd
import ast
import re

def time_to_24h(time_str):
    if not isinstance(time_str, str): return None, None
    time_str = time_str.replace('\u202f', ' ').strip().upper()
    match = re.search(r'(\d+)(?::(\d+))?\s*(AM|PM)?', time_str)
    if not match: return None, None
    hour, minute = int(match.group(1)), int(match.group(2)) if match.group(2) else 0
    period = match.group(3)
    if period == 'PM' and hour < 12: hour += 12
    if period == 'AM' and hour == 12: hour = 0
    return hour, minute

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Improved Open Hours Parsing
open_hours_rows = []
for _, row in df.iterrows():
    oh = row.get('open_hours')
    if not oh or pd.isna(oh): continue

    # Ensure we have a dict
    if isinstance(oh, str):
        try: oh = ast.literal_eval(oh)
        except: continue

    if isinstance(oh, dict):
        for day, intervals in oh.items():
            if not isinstance(intervals, list): continue
            for interval in intervals:
                interval = interval.replace('\u2013', '-').replace('\u2014', '-')
                if '-' in interval:
                    parts = interval.split('-')
                    s_h, s_m = time_to_24h(parts[0])
                    e_h, e_m = time_to_24h(parts[1])
                    if s_h is not None:
                        open_hours_rows.append((row['place_id'], day, s_h, s_m, e_h, e_m))

cursor.execute("DROP TABLE IF EXISTS open_hours_detailed")
cursor.execute("CREATE TABLE open_hours_detailed (place_id TEXT, day TEXT, start_hour INTEGER, start_minute INTEGER, end_hour INTEGER, end_minute INTEGER)")
if open_hours_rows:
    cursor.executemany("INSERT INTO open_hours_detailed VALUES (?,?,?,?,?,?)", open_hours_rows)
    print(f"Success! Inserted {len(open_hours_rows)} rows into open_hours_detailed.")

conn.commit()
conn.close()

### Download Consolidated Data
Run the cell below to download the final processed dataset to your local machine.

In [ ]:
# ### Final Export Status
# Files are now being consolidated into the `/nb_output` directory on Google Drive.
print("Initiating final consolidation...")

### Consolidating Outputs to Drive
This cell creates the `nb_output` directory and copies all generated artifacts into it for long-term storage.

In [ ]:
import shutil
import os

# output_dir is already defined and created earlier in the notebook.

output_dir = os.path.join(json_directory_path, 'nb_output') # Ensure output_dir is accessible and correct
os.makedirs(output_dir, exist_ok=True) # Ensure it exists

# Define the full paths to the files expected in the output directory
files_to_check = [
    os.path.join(output_dir, f'combined_{datasetName}s.csv'),
    os.path.join(output_dir, f'ktm_{datasetName}.db'),
    os.path.join(output_dir, 'dataset_summary.csv'),
    os.path.join(output_dir, 'dataset_summary.md')
]

print(f"--- Starting Final Backup Confirmation for {datasetName} to Drive ---")
for file_path in files_to_check:
    file_name = os.path.basename(file_path)
    if os.path.exists(file_path):
        print(f"Confirmed output file exists: {file_name}")
    else:
        print(f"Error: Output file not found for confirmation: {file_name}")